In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go

In [2]:
perfum_men = pd.read_csv(r"D:\projects\Perfume Ebay Dataset\ebay_mens_perfume.csv")
perfum_women = pd.read_csv(r"D:\projects\Perfume Ebay Dataset\ebay_womens_perfume.csv")

About Data

In [3]:
perfum_men["target_audience"] = "for men"
perfum_women["target_audience"] = "for women"
perfum = pd.concat([perfum_men , perfum_women] , ignore_index=True) 
perfum.head()

,brand,title,type,price,priceWithCurrency,available,availableText,sold,lastUpdated,itemLocation,target_audience
0,Dior,Christian Dior Sauvage Men's EDP 3.4 oz Fragra...,Eau de Parfum,84.99,US $84.99/ea,10.0,More than 10 available / 116 sold,116.0,"May 24, 2024 10:03:04 PDT","Allen Park, Michigan, United States",for men
1,AS SHOW,A-v-entus Eau de Parfum 3.3 oz 100ML Millesime...,Eau de Parfum,109.99,US $109.99,8.0,8 available / 48 sold,48.0,"May 23, 2024 23:07:49 PDT","Atlanta, Georgia, Canada",for men
2,Unbranded,HOGO BOSS cologne For Men 3.4 oz,Eau de Toilette,100.00,US $100.00,10.0,More than 10 available / 27 sold,27.0,"May 22, 2024 21:55:43 PDT","Dearborn, Michigan, United States",for men
3,Giorgio Armani,Acqua Di Gio by Giorgio Armani 6.7 Fl oz Eau D...,Eau de Toilette,44.99,US $44.99/ea,2.0,2 available / 159 sold,159.0,"May 24, 2024 03:30:43 PDT","Reinholds, Pennsylvania, United States",for men
4,Lattafa,Lattafa Men's Hayaati Al Maleky EDP Spray 3.4 ...,Fragrances,16.91,US $16.91,NaN,Limited quantity available / 156 sold,156.0,"May 24, 2024 07:56:25 PDT","Brooklyn, New York, United States",for men


In [4]:
perfum.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2000 entries, 0 to 1999
Data columns (total 11 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   brand              1998 non-null   object 
 1   title              2000 non-null   object 
 2   type               1995 non-null   object 
 3   price              2000 non-null   float64
 4   priceWithCurrency  2000 non-null   object 
 5   available          1758 non-null   float64
 6   availableText      1989 non-null   object 
 7   sold               1978 non-null   float64
 8   lastUpdated        1874 non-null   object 
 9   itemLocation       2000 non-null   object 
 10  target_audience    2000 non-null   object 
dtypes: float64(3), object(8)
memory usage: 172.0+ KB


In [5]:
perfum.describe(include='O').T

,count,unique,top,freq
brand,1998,401,Giorgio Armani,72
title,2000,1941,YSL Yves Saint Laurent Y Eau de Perfume Spray ...,4
type,1995,116,Eau de Parfum,847
priceWithCurrency,2000,1164,US $29.99/ea,38
availableText,1989,1300,More than 10 available / 2 sold,12
lastUpdated,1874,1831,"May 24, 2024 10:26:59 PDT",5
itemLocation,2000,447,"Dallas, Texas, United States",267
target_audience,2000,2,for men,1000


In [6]:
perfum.describe().T

,count,mean,std,min,25%,50%,75%,max
price,2000.0,43.187090,32.619625,1.99,21.9725,34.04,53.99,299.99
available,1758.0,20.728669,56.781389,2.00,5.0000,10.00,10.00,842.00
sold,1978.0,632.473711,2470.055822,1.00,14.0000,51.00,285.75,54052.00


# ***Data Cleaning*** 

# **Handling Missing Values**

In [7]:
perfum.isnull().sum()

brand                  2
title                  0
type                   5
price                  0
priceWithCurrency      0
available            242
availableText         11
sold                  22
lastUpdated          126
itemLocation           0
target_audience        0
dtype: int64

In [8]:
brand_list = perfum['brand'].dropna().unique().tolist()
for b in brand_list:
    mask = perfum['brand'].isnull() & perfum['title'].str.contains(b, case=False, na=False)
    perfum.loc[mask, 'brand'] = b

In [9]:
type_list = perfum['type'].dropna().unique().tolist()
for b in type_list:
    mask = perfum['type'].isnull() & perfum['title'].str.contains(b, case=False, na=False)
    perfum.loc[mask, 'type'] = b

C:\Users\Al-Braka\AppData\Local\Temp\ipykernel_11100\2126809846.py:3: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  mask = perfum['type'].isnull() & perfum['title'].str.contains(b, case=False, na=False)


In [10]:
av_text = perfum.loc[perfum['available'].isnull() , 'availableText']
av_text = av_text.str.split('/').str[0].str.strip()
av_text.value_counts()

availableText
Last One                                          120
Limited quantity available                        103
More than 10 lots available (3 items per lot)       2
Out of Stock                                        2
More than 10 lots available (10 items per lot)      1
3 lots available (10 items per lot)                 1
2 disponibles                                       1
4 lots available (8 items per lot)                  1
Name: count, dtype: int64

In [11]:
perfum.loc[perfum['available'].isnull() & perfum['availableText'].str.contains('last one', case=False, na=False), 'available'] = 1

perfum.loc[perfum['available'].isnull() & perfum['availableText'].str.contains('limited quantity available', case=False, na=False), 'available'] = 2

perfum.loc[perfum['available'].isnull() & perfum['availableText'].str.contains('lots available ', case=False, na=False), 'available'] = 30

perfum.loc[perfum['available'].isnull() & perfum['availableText'].str.contains('disponibles', case=False, na=False), 'available'] = 2

perfum.loc[perfum['available'].isnull() & perfum['availableText'].str.contains('Out of Stock', case=False, na=False), 'available'] = 0


In [12]:
av_text = perfum.loc[perfum['sold'].isnull() , 'availableText']
av_text.value_counts()

availableText
More than 10 available        5
4 available                   2
10 available                  1
2 available                   1
5 available                   1
2 disponibles / 3 vendidos    1
Name: count, dtype: int64

In [13]:
perfum.loc[perfum['sold'].isnull() & perfum['availableText'].notnull() , 'sold'] = 0

In [14]:
perfum.dropna(subset=['available', 'availableText' , 'sold'] , how='all' , inplace=True)

In [15]:
perfum.isnull().sum()

brand                  0
title                  0
type                   0
price                  0
priceWithCurrency      0
available              0
availableText          0
sold                   0
lastUpdated          115
itemLocation           0
target_audience        0
dtype: int64

# **Handling duplicated Values**

In [16]:
perfum.duplicated().sum()

np.int64(1)

In [17]:
perfum.drop_duplicates(inplace=True)

# **Handling Inconsistant Value**

In [18]:
perfum_n = perfum.copy()

In [19]:
list_lower = ['brand' , 'title' ,'type' , 'availableText' ,'itemLocation' ]
for n in list_lower:
 perfum_n[n] = perfum_n[n].str.strip().str.lower()

In [20]:
clean_dict = {
    'as photos': 'as shown',
    'as show': 'as shown',
    'as picture show': 'as shown',
    'as showed': 'as shown',
    'as picture shown': 'as shown',
    'tiffany & co.': 'tiffany',
    'polo ralph lauren': 'polo',
    'pink': 'pink sugar',
    'parfums gres': 'perfume',
    'parfums grÃ¨s': 'perfume',
    'parfums': 'perfume',
    'parfum': 'perfume',
    'michael malul london': 'michael malul',
    'michael malul gents scents': 'michael malul',
    'mercedes-benz': 'mercedes benz',
    'maison martin margiela': 'maison margiela',
    'lattafa perfumes': 'lattafa',
    'lancÃ´me': 'lancome',
    'lâ€™occitane': 'lacoste',
    'king of kings': 'king',
    'kenneth cole reaction': 'kenneth cole',
    'kate spade new york': 'kate spade',
    'huda beauty kayali': 'huda beauty',
    'hermÃˆs ': 'hermes',
    'have a scent': 'heaven scents',
    'guerlain paris': 'guerlain',
    'giorgioÂ² armani': 'giorgio armani',
    'giorgio arm.ani': 'giorgio armani',
    'giorgi^o armani': 'giorgio armani',
    'fragrance world': 'fragrance',
    'fragrance couture': 'fragrance',
    'fragonard': 'fragrance',
    'fragance one': 'fragrance',
    'estÃ©e lauder': 'estee lauder',
    'elizabeth and james nirvana': 'elizabeth & james',
    'dolce&gabbana': 'dolce & gabbana',
    'dolce gabbana': 'dolce & gabbana',
    'coty inc.': 'coty',
    'chloÃ©': 'chloe',
    'change for women': 'chanel'
}
perfum_n['brand'] = perfum_n['brand'].replace(clean_dict)

In [21]:
perfum_n.loc[perfum_n['type'].str.contains('/', case=False, na=False), 'type'] = 'eau de parfum'
perfum_n.loc[perfum_n['type'].str.contains('1', case=False, na=False), 'type'] = 'eau de parfum'
perfum_n.loc[perfum_n['type'].str.contains('eau de parfum', case=False, na=False), 'type'] = 'eau de parfum'
perfum_n.loc[perfum_n['type'].str.contains('edp', case=False, na=False), 'type'] = 'eau de parfum'
perfum_n.loc[perfum_n['type'].str.contains('eau de to', case=False, na=False), 'type'] = 'eau de toilette'
perfum_n.loc[perfum_n['type'].str.contains('edt', case=False, na=False), 'type'] = 'eau de toilette'
perfum_n.loc[perfum_n['type'].str.contains('eau de Cologne', case=False, na=False), 'type'] = 'eau de Cologne'
perfum_n.loc[perfum_n['type'].str.contains('edc', case=False, na=False), 'type'] = 'eau de Cologne'
perfum_n.loc[perfum_n['type'].str.contains('colog', case=False, na=False), 'type'] = 'cologne'
perfum_n.loc[perfum_n['type'].str.contains('cologne', case=False, na=False), 'type'] = 'cologne'
perfum_n.loc[perfum_n['type'].str.contains('elix', case=False, na=False), 'type'] = 'elixer de parfum'
perfum_n.loc[perfum_n['type'].str.contains('extrait', case=False, na=False), 'type'] = 'extrait de Parfum'
perfum_n.loc[perfum_n['type'].str.contains('oil', case=False, na=False), 'type'] = 'oil'


In [22]:
dic = {'perfume':'eau de perfume' , 'parfum':'eau de perfume'}
perfum_n['type'] = perfum_n['type'].replace(dic)

In [23]:
perfum_n['type'].unique()

array(['eau de parfum', 'oil', 'fragrances', 'eau de perfume',
       'le parfum', 'unscented', 'cologne', 'extrait de Parfum',
       'pheromone', 'aftershave', 'fragrance & perfume', 'y', 'gift sets',
       'fragrance rolling ball', 'body spray', 'does not apply',
       'editions parfums', 'deodorant', 'de nuit', 'parfum intense',
       'roll on', 'elixer de parfum', 'various', 'assorted',
       'deodorant body spray', 'splash-on', 'car air freshener',
       'fragrance body spray', 'spray',
       '~ body firm advanced body repair treatment ~', 'fragrance mist',
       'deodorant stick', '3 pc', 'mist', 'hair perfume', 'cream',
       'skin_moisturizer', 'fine fragrance mist', 'lotion', 'shimmer',
       'perfume fragrance mist', 'parfum, lotion, gloss and blush',
       'body mist', 'asst', 'beauty', 'extract parfum', 'body powder',
       'perfume gift sets', 'body lotion', 'esprit de parfum',
       'solid perfume stick', 'sensuous body moisturizer', 'pink sugar',
       'fra

In [24]:
perfum_n.describe(include='O').T

,count,unique,top,freq
brand,1988,337,dolce & gabbana,86
title,1988,1926,idole by lancome eau de parfum edp perfume for...,4
type,1988,54,eau de parfum,900
priceWithCurrency,1988,1158,US $29.99/ea,38
availableText,1988,1300,more than 10 available / 2 sold,12
lastUpdated,1874,1831,"May 24, 2024 10:26:59 PDT",5
itemLocation,1988,440,"dallas, texas, united states",267
target_audience,1988,2,for men,997


In [25]:
perfum_n.info()

<class 'pandas.core.frame.DataFrame'>
Index: 1988 entries, 0 to 1999
Data columns (total 11 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   brand              1988 non-null   object 
 1   title              1988 non-null   object 
 2   type               1988 non-null   object 
 3   price              1988 non-null   float64
 4   priceWithCurrency  1988 non-null   object 
 5   available          1988 non-null   float64
 6   availableText      1988 non-null   object 
 7   sold               1988 non-null   float64
 8   lastUpdated        1874 non-null   object 
 9   itemLocation       1988 non-null   object 
 10  target_audience    1988 non-null   object 
dtypes: float64(3), object(8)
memory usage: 186.4+ KB


# **feature engneering**

In [26]:
perfum_n['location_parts'] = perfum_n['itemLocation'].str.split(',').apply(lambda x: [p.strip() for p in x])

In [27]:
perfum_n['city'] = perfum_n['location_parts'].apply(lambda x: x[0] if len(x) > 0 else None)
perfum_n['state'] = perfum_n['location_parts'].apply(lambda x: x[1] if len(x) == 3 else None)
perfum_n['country'] = perfum_n['location_parts'].apply(lambda x: x[-1] if len(x) > 0 else None)

In [28]:
perfum_n['country'].unique()

array(['united states', 'canada', 'china', 'hong kong', 'taiwan',
       'israel', 'poland', 'brazil', 'portugal', 'india', 'japan',
       'estados unidos', 'pakistan', 'bulgaria'], dtype=object)

In [29]:
country_cleaning = {
    'hong kong':'china',
    'israel': 'Palestine',
}
perfum_n['country'] = perfum_n['country'].replace(country_cleaning)

In [36]:
perfum_n.drop(columns=['location_parts'])

,brand,title,type,price,priceWithCurrency,available,availableText,sold,lastUpdated,itemLocation,target_audience,city,state,country
0,dior,christian dior sauvage men's edp 3.4 oz fragra...,eau de parfum,84.99,US $84.99/ea,10.0,more than 10 available / 116 sold,116.0,"May 24, 2024 10:03:04 PDT","allen park, michigan, united states",for men,allen park,michigan,united states
1,as shown,a-v-entus eau de parfum 3.3 oz 100ml millesime...,eau de parfum,109.99,US $109.99,8.0,8 available / 48 sold,48.0,"May 23, 2024 23:07:49 PDT","atlanta, georgia, canada",for men,atlanta,georgia,canada
2,unbranded,hogo boss cologne for men 3.4 oz,oil,100.00,US $100.00,10.0,more than 10 available / 27 sold,27.0,"May 22, 2024 21:55:43 PDT","dearborn, michigan, united states",for men,dearborn,michigan,united states
3,giorgio armani,acqua di gio by giorgio armani 6.7 fl oz eau d...,oil,44.99,US $44.99/ea,2.0,2 available / 159 sold,159.0,"May 24, 2024 03:30:43 PDT","reinholds, pennsylvania, united states",for men,reinholds,pennsylvania,united states
4,lattafa,lattafa men's hayaati al maleky edp spray 3.4 ...,fragrances,16.91,US $16.91,2.0,limited quantity available / 156 sold,156.0,"May 24, 2024 07:56:25 PDT","brooklyn, new york, united states",for men,brooklyn,new york,united states
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1995,avon,avon far away infinity eau de parfum 1.7 fl. o...,eau de parfum,13.89,US $13.89,10.0,more than 10 available / 157 sold,157.0,"May 16, 2024 22:35:29 PDT","west palm beach, florida, united states",for women,west palm beach,florida,united states
1996,mancera,roses greedy by mancera perfume for unisex edp...,eau de parfum,57.85,US $57.85/ea,33.0,33 available / 58 sold,58.0,"May 24, 2024 08:03:11 PDT","dallas, texas, united states",for women,dallas,texas,united states
1997,unbranded,"sweet tooth eau de parfum, perfume for women, ...",eau de parfum,30.96,US $30.96,2.0,2 available / 3 sold,3.0,"May 17, 2024 23:16:41 PDT","new york, new york, united states",for women,new york,new york,united states
1998,juliette has a gun,mmmm by juliette has a gun perfume for her edp...,eau de perfume,53.99,US $53.99/ea,3.0,3 available / 117 sold,117.0,"May 13, 2024 22:19:34 PDT","dallas, texas, united states",for women,dallas,texas,united states


In [35]:
perfum_n.head()

,brand,title,type,price,priceWithCurrency,available,availableText,sold,lastUpdated,itemLocation,target_audience,location_parts,city,state,country
0,dior,christian dior sauvage men's edp 3.4 oz fragra...,eau de parfum,84.99,US $84.99/ea,10.0,more than 10 available / 116 sold,116.0,"May 24, 2024 10:03:04 PDT","allen park, michigan, united states",for men,"[allen park, michigan, united states]",allen park,michigan,united states
1,as shown,a-v-entus eau de parfum 3.3 oz 100ml millesime...,eau de parfum,109.99,US $109.99,8.0,8 available / 48 sold,48.0,"May 23, 2024 23:07:49 PDT","atlanta, georgia, canada",for men,"[atlanta, georgia, canada]",atlanta,georgia,canada
2,unbranded,hogo boss cologne for men 3.4 oz,oil,100.00,US $100.00,10.0,more than 10 available / 27 sold,27.0,"May 22, 2024 21:55:43 PDT","dearborn, michigan, united states",for men,"[dearborn, michigan, united states]",dearborn,michigan,united states
3,giorgio armani,acqua di gio by giorgio armani 6.7 fl oz eau d...,oil,44.99,US $44.99/ea,2.0,2 available / 159 sold,159.0,"May 24, 2024 03:30:43 PDT","reinholds, pennsylvania, united states",for men,"[reinholds, pennsylvania, united states]",reinholds,pennsylvania,united states
4,lattafa,lattafa men's hayaati al maleky edp spray 3.4 ...,fragrances,16.91,US $16.91,2.0,limited quantity available / 156 sold,156.0,"May 24, 2024 07:56:25 PDT","brooklyn, new york, united states",for men,"[brooklyn, new york, united states]",brooklyn,new york,united states


In [37]:
perfum_n.to_csv(r'D:\projects\Perfume Ebay Dataset\perfum_cleaned.csv')